# Document Pre-processing for Knowledge Tuning

## Overview

This notebook demonstrates a complete document preprocessing pipeline designed specifically for **knowledge tuning** with sdg-hub. 

## What This Notebook Does

This preprocessing pipeline transforms raw documents (PDFs, Word docs, etc.) into seed data for data generation:

1. **Document Parsing**: Converts raw documents to structured markdown format
2. **Chunking**: Splits documents into manageable chunks while preserving structure and context
3. **Seed Data Creation**: Formats chunks with in-context learning (ICL) templates for effective knowledge tuning

## Prerequisites

- We will use the existing InstructLab document parser (`docparser_v2.py`) and Document parsing configuration (`docling_v2_config.yaml`)
- Raw pdf documents in the `document_collection/` directory


In [ ]:
# Step 1: Document Processing Pipeline
# Define the directory containing raw documents to be processed
data_dir = "document_collection/"

# Run the document parser to convert documents to markdown
# - input-dir: Directory containing source documents
# - output-dir: Directory where processed markdown files will be saved
# - c: Configuration file specifying parsing parameters
!python ../docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../docling_v2_config.yaml

In [ ]:
# Step 2: Install Required Dependencies
# Install packages needed for document processing and text chunking

%pip install docling markdown-it-py
%pip install --upgrade transformers

In [ ]:
# Step 3: Load Processed Documents
import glob
import os
from pathlib import Path

# In our example above docling step produces markdown of all the PDF files in document_collection
md_files = sorted(glob.glob(f"{data_dir}/*.md"))

if not md_files:
    raise ValueError(f"No markdown files found in {data_dir}. Run Step 1 first.")

documents = []
for md_path in md_files:
    with open(md_path, "r", encoding="utf-8") as f:
        documents.append(
            {
                "source_file": os.path.basename(md_path),
                "document_outline": Path(md_path).stem,
                "text": f.read(),
            }
        )

print(f"Loaded {len(documents)} markdown files from {data_dir}")
for doc in documents:
    print(f"  - {doc['source_file']}")


In [ ]:
# Step 4: Text Chunking and Dataset Creation

from markdown_it import MarkdownIt
from typing import List
import datasets


def chunk_markdown(text: str, max_tokens: int = 200, overlap: int = 50) -> List[str]:
    """
    Splits Markdown text into chunks at block-level elements
    (headings, paragraphs, lists, tables, code, blockquotes).
    Adds overlap (in words) between all consecutive chunks.

    Args:
        text: The markdown text to be chunked
        max_tokens: Maximum number of words per chunk
        overlap: Number of overlapping words between consecutive chunks

    Returns:
        List of text chunks with specified overlap
    """

    # Initialize markdown parser to understand document structure
    md = MarkdownIt()
    tokens = md.parse(text)

    # Group tokens into block-level segments to preserve markdown structure
    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
                buf = []
        elif tok.content:
            buf.append(tok.content)
    if buf:
        blocks.append("\n".join(buf).strip())

    # Split blocks into chunks with overlap to maintain context continuity
    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                chunks.append(" ".join(current_words))
                current_words = current_words[-overlap:] if overlap > 0 else []

    # Add any remaining words as the final chunk
    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


all_rows = []
for doc in documents:
    doc_chunks = chunk_markdown(doc["text"], max_tokens=5000, overlap=1000)
    for chunk in doc_chunks:
        all_rows.append(
            {
                "document": chunk,
                "document_outline": doc["document_outline"],
                "source_file": doc["source_file"],
            }
        )

if not all_rows:
    raise ValueError("No chunks were produced from markdown documents.")

seed_data = datasets.Dataset.from_list(all_rows)

# Required ICL fields for the SDG-Hub flow
example_doc = all_rows[0]["document"]
example_doc = " ".join(example_doc.split()[:250])
domain = os.getenv("SEED_DOMAIN", "general")

icl = {
    "icl_document": example_doc,
    "icl_query_1": "What are the most important points presented in this document section?",
    "icl_query_2": "How do the ideas in this section connect to the document's main objective?",
    "icl_query_3": "What practical implications can be derived from this section?",
    "domain": domain,
}

seed_data = seed_data.map(lambda x: icl)

seed_data_path = os.getenv("SEED_DATA_PATH", "seed_data.jsonl")
seed_data.to_json(seed_data_path, orient="records", lines=True)

print(f"Saved seed data to: {seed_data_path}")
print(f"Total chunks: {len(seed_data)}")
print(f"Columns: {seed_data.column_names}")


### Next Steps:
- The seed_data.jsonl file is now ready for the knowledge tuning pipeline.
- You can now refer to the [knowledge generation](knowledge_generation.ipynb) notebook